In [ ]:
%reload_ext autoreload
%autoreload 2
%cd ~/erc-src/cuneiform-ocr-sign-alignment-worktree
%env PATH=$HOME/.local/bin:$PATH

import os
import cv2
import numpy as np
import torch
from dotenv import load_dotenv

from sign_alignment.detector import ModelConfig, TabletImageDetector
from sign_alignment.data_source import LocalDataSource
from sign_alignment.visualizer import ColorConfig

ANNOTATIONS_DIR = os.path.expanduser("~/erc-work-data/data-of-cuneiform-ocr-data/filtered_annotations")
CONFIG_FILE = "configs/detr.py"
CHECKPOINT_FILE = os.path.expanduser("~/erc-work-data/retrained_models/detr-173/epoch_1000.pth")
# # temporal change
# CHECKPOINT_FILE = os.path.expanduser("~/epoch_1000.pth")
# ANNOTATIONS_DIR = os.path.expanduser("~/filtered_annotations")
# # ---
SCORE_THRESHOLD = 0.5
OUTPUT_DIR = "alignment_results"
SAMPLE_LIMIT = 5

load_dotenv()
MONGODB_URI = os.getenv('MONGODB_URI', 'YOUR_MONGODB_URI')
CANONICAL_FEATURE_DIR = "~/erc-work-data/signs_alignment_data/precompute_feautures/"
if not MONGODB_URI or MONGODB_URI == 'YOUR_MONGODB_URI':
    raise ValueError("MONGODB_URI is required for Mongo-backed canonical sign images")



# --- show ipynb kernel connection info ---
import sys
from pathlib import Path

from ipykernel.connect import get_connection_file
from jupyter_core.paths import jupyter_runtime_dir

runtime_dir = Path(jupyter_runtime_dir()).expanduser()
kernel_file = Path(get_connection_file()).expanduser()
if not kernel_file.is_absolute():
    kernel_file = runtime_dir / kernel_file
kernel_file = kernel_file.resolve()
jupyter_bin = Path(sys.executable).with_name("jupyter")

print("runtime dir:", runtime_dir)
print("kernel file:", kernel_file)
print("connect cmd:", f"{jupyter_bin} console --existing {kernel_file}")

In [ ]:
from sign_alignment.pipeline import CropContext, Runner, VisOptions
from sign_alignment.data_source import EBLMongoCanonicalSource, PrototypeSource
from sign_alignment.dift_model import DiftConfig, DiftModel
from sign_alignment.dift_align import DiftAlignmentConfig, DiftRuntime

_DIFT_REPO = os.path.expanduser("~/erc-src/ProtoSnap")
canonical_source = EBLMongoCanonicalSource(MONGODB_URI)
proto_src = PrototypeSource()
dift = DiftRuntime(
    model=DiftModel(DiftConfig(repo_root=_DIFT_REPO)),
    feature_dir=CANONICAL_FEATURE_DIR,
    config=DiftAlignmentConfig(affine_probe_padding_ratio=0.1),
)

model_config = ModelConfig(
    config_file=CONFIG_FILE,
    checkpoint_file=CHECKPOINT_FILE,
    device='auto'
)

if 'tablet_detector' not in globals() or getattr(tablet_detector, 'model', None) is None:
    tablet_detector = TabletImageDetector(
        score_threshold=SCORE_THRESHOLD,
        model_config=model_config,
        keep_crops=True,
        is_crop_itself=False,
    )
else:
    print("Reusing existing tablet_detector instance.")

crop_context = CropContext(
    tablet_detector=tablet_detector,
    local_source=LocalDataSource(ANNOTATIONS_DIR),
    color_config=ColorConfig,
    output_dir=OUTPUT_DIR,
    img_idx=1,
    dift=dift,
    canonical_source=proto_src,
)

vis = VisOptions(info=True, display=True, save=True)

runner = Runner(context=crop_context, vis=vis)


In [ ]:
from sign_alignment.data_source import PrototypeSource
src_proto = PrototypeSource()

img = src_proto.get("AN", 'Neo Assyrian')
import matplotlib.pyplot as plt
plt.imshow(img, cmap="gray", vmin=0, vmax=255)
plt.axis('off')
plt.show() # Try image without background

In [ ]:

# periods for selected fragments
check_periods = runner._fragments[0:20]

periods = []
for fragment_id in check_periods:
    fragment_data = runner.context.api_source.get_fragment_data(fragment_id)
    periods.append({
        "fragment": fragment_id,
        "period": (fragment_data or {}).get("script", {}).get("period", "<missing>"),
    })

periods

In [ ]:
import sign_alignment.pipeline as pp

runner.choose_sample(9)  # index 0 = NBC.4020, index 9 = HS.2086
# Load image, ground truth, and sign text from API in one step
runner.choose_sample(name="YBC.12860")  # switch sample here
runner.choose_sample(name="ND.5437")  # switch sample here
runner.run([pp.Step("Load data", pp.load_data, pp.vis_loaded_data)])


In [ ]:
# detect signs (full image + chosen exp_image crop)
runner.choose_crop(1)  # switch crop index here (0 = full image, 1+ = exp_image crops)
runner.run([pp.Step("Detect signs", pp.detect_signs, pp.vis_detections)])

In [ ]:
# transform GT boxes into sub-image coordinates and visualize
runner.run([pp.Step("Transform GT to crop", pp.transform_gt_to_crop, pp.vis_crop_ground_truth)])


In [ ]:
# compute average detection box dimensions
runner.run([pp.Step("Detection statistics", lambda _: None, pp.vis_detection_statistics)])

In [ ]:
# create detection and text sub-tablets
runner.run([pp.Step("Create box sets", pp.create_box_sets, pp.vis_box_sets)])

In [ ]:
# DBSCAN row detection on detection sub-tablet (also reports text sub-tablet rows)
runner.run([pp.Step("Detect rows", pp.detect_rows, pp.vis_detected_rows_info)])

In [ ]:
# DP row matching between detection and text sub-tablets
runner.run([pp.Step("Match rows", pp.match_rows, pp.vis_row_matches)])

In [ ]:
# visualize detection rows with D# / D#→R# labels
runner.run([pp.Step("Visualize detection rows", lambda _: None, pp.vis_detection_rows)])

In [ ]:
# within-row sign matching for each matched row pair
runner.run([pp.Step("Match signs", pp.match_signs_in_rows, pp.vis_sign_matches)])

In [ ]:
# align text rows onto detection rows using regression baselines
# result stored in aligned_boxes
runner.run([pp.Step("Align text rows", pp.align_text_rows, pp.vis_aligned_rows)])


In [ ]:
# build sign match info; draw text mapping, side-by-side composite, alignment diagnostic
runner.run([pp.Step("Build sign match info", pp.build_sign_match_info, pp.vis_sign_match_info)])

In [ ]:
# position offset analysis: coarse-aligned vs detection boxes
runner.run([pp.Step("Offset analysis", lambda _: None, pp.vis_offset_analysis)])


In [ ]:
# unload detector model from GPU to free VRAM before DIFT / PSR optimization
runner.run([pp.Step("Unload detector", pp.unload_detector)])  # optional

In [ ]:
# Canonical setup creates a period-specific feature cache; images are queried on demand

# runner.run([pp.Step("Setup canonical signs", pp.setup_canonical_signs, pp.vis_canonical_signs)])
runner.run([pp.Step("Setup canonical signs", pp.setup_canonical_signs)]) # skip vis now

In [ ]:
# DIFT sampling-grid scores on the crop tablet
runner.run([pp.Step("DIFT sampling grid scores", lambda _: None, pp.vis_dift_score_on_whole_tablet)])

In [ ]:
# Select a crop by x1, y1, width, height and compare it with a canonical sign using DIFT features
runner.run([pp.Step("Manual DIFT crop match", lambda _: None, pp.vis_manual_dift_crop_match)])

In [ ]:
# create PSR optimizer and plot characteristic loss curves
runner.run([pp.Step("Create PSR optimizer", pp.create_psr_optimizer, pp.vis_psr_optimizer)])

In [ ]:
# run PSR optimization, visualize canonical signs at current boxes, probe current boxes, then continue to final
runner.run([
    pp.Step("Optimize until DIFT probe", pp.optimize_psr_until_dift_probe, pp.vis_optimization),
    pp.Step("Canonical sign overlay", pp.create_canonical_sign_overlay, pp.vis_canonical_sign_overlay),
    pp.Step("DIFT affine probe", pp.run_dift_affine_probe, pp.vis_dift_affine_probe),
    pp.Step("Finish PSR optimization", pp.optimize_psr_after_dift_probe, pp.vis_optimization),
])

In [ ]:
# optimization loss history
runner.run([pp.Step("Plot loss history", lambda _: None, pp.vis_loss_history)])

In [ ]:
# 2x2 results comparison: coarse aligned, final optimized, det+final overlay, gt+final overlay
runner.run([pp.Step("Results comparison", lambda _: None, pp.vis_results_comparison)])

In [ ]:
# analyze parameter changes between coarse-aligned and final optimized
runner.run([pp.Step("Parameter changes", lambda _: None, pp.vis_parameter_changes)])